In [1]:
import re
from collections import defaultdict
from typing import List, Tuple, Dict, Set

In [4]:
class PlagiarismDetector:
    """
    Plagiarism detection system using the Rabin-Karp algorithm.
    Compares documents by hashing sequences of words.
    """
    
    def __init__(self, d=256, q=101):
        """
        Initialize the plagiarism detector.
        
        Args:
            d: Base for hashing (default 256)
            q: Prime number for modulo operation (default 101)
        """
        self.d = d
        self.q = q
    
    def _compute_hash(self, text: str, length: int) -> int:
        """Compute hash value for a text segment."""
        hash_val = 0
        for i in range(length):
            hash_val = (self.d * hash_val + ord(text[i])) % self.q
        return hash_val
    
    def tokenize_text(self, text: str) -> List[str]:
        """
        Tokenize text into words, removing punctuation and converting to lowercase.
        """
        # Remove punctuation and convert to lowercase
        text = re.sub(r'[^\w\s]', ' ', text.lower())
        # Split into words and filter out empty strings
        words = [word for word in text.split() if word]
        return words
    
    def detect_plagiarism(self, document1: str, document2: str, 
                         window_size: int = 5, 
                         threshold: float = 0.3) -> Dict:
        """
        Detect plagiarism between two documents by finding matching word sequences.
        
        Args:
            document1: First document text
            document2: Second document text
            window_size: Number of consecutive words to compare (n-gram size)
            threshold: Similarity threshold (0-1) to flag as plagiarism
        
        Returns:
            Dictionary containing similarity score and matching segments
        """
        # Tokenize documents
        words1 = self.tokenize_text(document1)
        words2 = self.tokenize_text(document2)
        
        if len(words1) < window_size or len(words2) < window_size:
            return {
                'similarity_score': 0.0,
                'is_plagiarism': False,
                'matching_segments': [],
                'details': 'Documents too short for comparison'
            }
        
        # Create n-grams and their hashes for document 2
        doc2_hashes = defaultdict(list)
        
        for i in range(len(words2) - window_size + 1):
            ngram = ' '.join(words2[i:i + window_size])
            hash_val = self._compute_hash(ngram, len(ngram))
            doc2_hashes[hash_val].append((i, ngram))
        
        # Find matches in document 1
        matches = []
        matched_positions = set()
        
        for i in range(len(words1) - window_size + 1):
            ngram = ' '.join(words1[i:i + window_size])
            hash_val = self._compute_hash(ngram, len(ngram))
            
            # Check if hash exists in document 2
            if hash_val in doc2_hashes:
                for doc2_pos, doc2_ngram in doc2_hashes[hash_val]:
                    # Verify actual match (handle hash collisions)
                    if ngram == doc2_ngram and i not in matched_positions:
                        matches.append({
                            'doc1_position': i,
                            'doc2_position': doc2_pos,
                            'matched_text': ngram,
                            'word_count': window_size
                        })
                        matched_positions.add(i)
                        break
        
        # Calculate similarity score
        total_ngrams = len(words1) - window_size + 1
        similarity_score = len(matches) / total_ngrams if total_ngrams > 0 else 0
        
        return {
            'similarity_score': round(similarity_score, 4),
            'is_plagiarism': similarity_score >= threshold,
            'matching_segments': matches,
            'total_matches': len(matches),
            'doc1_word_count': len(words1),
            'doc2_word_count': len(words2)
        }
    
    def compare_multiple_documents(self, documents: Dict[str, str], 
                                  window_size: int = 5,
                                  threshold: float = 0.3) -> List[Dict]:
        """
        Compare multiple documents for plagiarism detection.
        
        Args:
            documents: Dictionary with document names as keys and text as values
            window_size: N-gram size for comparison
            threshold: Similarity threshold
        
        Returns:
            List of plagiarism matches between document pairs
        """
        doc_names = list(documents.keys())
        results = []
        
        for i in range(len(doc_names)):
            for j in range(i + 1, len(doc_names)):
                doc1_name = doc_names[i]
                doc2_name = doc_names[j]
                
                comparison = self.detect_plagiarism(
                    documents[doc1_name],
                    documents[doc2_name],
                    window_size,
                    threshold
                )
                
                if comparison['is_plagiarism']:
                    results.append({
                        'document1': doc1_name,
                        'document2': doc2_name,
                        'similarity_score': comparison['similarity_score'],
                        'matches': comparison['total_matches']
                    })
        
        return sorted(results, key=lambda x: x['similarity_score'], reverse=True)

In [5]:
if __name__ == "__main__":
    detector = PlagiarismDetector()
    
    print("=" * 70)
    print("PLAGIARISM DETECTION SYSTEM - RABIN-KARP ALGORITHM")
    print("=" * 70)
    
    # Example: Compare two essays
    essay1 = """
    Climate change represents one of the most significant challenges facing 
    humanity today. The rising global temperatures are causing ice caps to melt 
    and sea levels to rise. Scientists agree that human activities are the 
    primary cause of recent climate change. We must take action now to reduce 
    carbon emissions and protect our planet for future generations.
    """
    
    essay2 = """
    The rising global temperatures are causing ice caps to melt and sea levels 
    to rise dramatically. This phenomenon represents a major threat to coastal 
    cities. Scientists agree that human activities are the primary cause of 
    recent climate change and we need immediate action. Renewable energy 
    sources offer hope for a sustainable future.
    """
    
    print("\n--- Two Document Comparison ---")
    result = detector.detect_plagiarism(essay1, essay2, window_size=5, threshold=0.2)
    
    print(f"\nDocument 1 Word Count: {result['doc1_word_count']}")
    print(f"Document 2 Word Count: {result['doc2_word_count']}")
    print(f"Similarity Score: {result['similarity_score']:.2%}")
    print(f"Plagiarism Detected: {result['is_plagiarism']}")
    print(f"Total Matching Segments: {result['total_matches']}")
    
    if result['matching_segments']:
        print(f"\nMatching Text Segments:")
        for i, match in enumerate(result['matching_segments'][:5], 1):
            print(f"  {i}. '{match['matched_text']}'")
        if len(result['matching_segments']) > 5:
            print(f"  ... and {len(result['matching_segments']) - 5} more matches")
    
    print("\n" + "=" * 70)
    print("--- Multiple Document Comparison ---")
    print("=" * 70)
    
    # Example: Compare multiple student submissions
    documents = {
        "Student A": """
        The water cycle is a continuous process where water evaporates from 
        oceans and lakes into the atmosphere. Then it condenses into clouds 
        and falls back to earth as precipitation. This cycle is essential 
        for life on earth.
        """,
        "Student B": """
        Water evaporates from oceans and lakes into the atmosphere where it 
        condenses into clouds and falls back to earth as precipitation. The 
        water cycle is a continuous process that is essential for all living 
        things on our planet.
        """,
        "Student C": """
        Photosynthesis is the process by which plants convert sunlight into 
        energy. This process is crucial for plant growth and produces oxygen 
        as a byproduct which is vital for animal life.
        """
    }
    
    comparisons = detector.compare_multiple_documents(
        documents, 
        window_size=4, 
        threshold=0.25
    )
    
    if comparisons:
        print("\nPlagiarism detected between the following document pairs:\n")
        for comp in comparisons:
            print(f"  {comp['document1']} ↔ {comp['document2']}")
            print(f"    Similarity: {comp['similarity_score']:.2%}")
            print(f"    Matching segments: {comp['matches']}\n")
    else:
        print("\nNo significant plagiarism detected among the documents.")
    
    print("=" * 70)
    print("\nTips for adjusting detection:")
    print("  - window_size: Larger values (7-10) detect longer phrases")
    print("  - window_size: Smaller values (3-5) are more sensitive")
    print("  - threshold: Higher values (0.4-0.6) reduce false positives")
    print("  - threshold: Lower values (0.2-0.3) catch more similarities")

PLAGIARISM DETECTION SYSTEM - RABIN-KARP ALGORITHM

--- Two Document Comparison ---

Document 1 Word Count: 56
Document 2 Word Count: 52
Similarity Score: 38.46%
Plagiarism Detected: True
Total Matching Segments: 20

Matching Text Segments:
  1. 'the rising global temperatures are'
  2. 'rising global temperatures are causing'
  3. 'global temperatures are causing ice'
  4. 'temperatures are causing ice caps'
  5. 'are causing ice caps to'
  ... and 15 more matches

--- Multiple Document Comparison ---

Plagiarism detected between the following document pairs:

  Student A ↔ Student B
    Similarity: 52.94%
    Matching segments: 18


Tips for adjusting detection:
  - window_size: Larger values (7-10) detect longer phrases
  - window_size: Smaller values (3-5) are more sensitive
  - threshold: Higher values (0.4-0.6) reduce false positives
  - threshold: Lower values (0.2-0.3) catch more similarities
